In [1]:
import os
import sys
import pandas as pd

sys.path.append(os.path.abspath('..'))

from etl.cleaning import *
from etl.nulls import *
from etl.normalization import *
from etl.schema import *

os.makedirs('../data/processed/nacimientos', exist_ok=True)
os.makedirs('../data/processed/fetales', exist_ok=True)
os.makedirs('../data/processed/no_fetales', exist_ok=True)


## 1. Nacimientos

In [2]:
rutaDatos = '../data/raw/nac2021.csv'
try:
    df_nac = pd.read_csv(rutaDatos, sep=',', encoding='latin-1')
    print(f"Archivo encontrado: \n")
    df_nac.info()
except FileNotFoundError:
    print(f"Archivo no encontrado: {rutaDatos}")

Archivo encontrado: 

<class 'pandas.DataFrame'>
RangeIndex: 616914 entries, 0 to 616913
Data columns (total 39 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   COD_DPTO        616914 non-null  int64  
 1   COD_MUNIC       616914 non-null  int64  
 2   AREANAC         616914 non-null  int64  
 3   SIT_PARTO       616914 non-null  int64  
 4   OTRO_SIT        1444 non-null    str    
 5   SEXO            616914 non-null  int64  
 6   PESO_NAC        616914 non-null  int64  
 7   TALLA_NAC       616914 non-null  int64  
 8   ANO             616914 non-null  int64  
 9   MES             616914 non-null  int64  
 10  ATEN_PAR        616914 non-null  int64  
 11  T_GES           616914 non-null  int64  
 12  T_GES_AGRU_CIE  616914 non-null  int64  
 13  NUMCONSUL       616914 non-null  int64  
 14  TIPO_PARTO      616914 non-null  int64  
 15  MUL_PARTO       616914 non-null  int64  
 16  APGAR1          616914 non-null  int64  
 17 

In [3]:
encontrar_duplicados(df_nac) # Encontrar filas duplicadas


===== ANÁLISIS DE DUPLICADOS =====
Total de registros: 616914
Filas duplicadas (incluyendo repetidas): 1337
Duplicados únicos a eliminar: 698


,COD_DPTO,COD_MUNIC,AREANAC,SIT_PARTO,OTRO_SIT,SEXO,PESO_NAC,TALLA_NAC,ANO,MES,...,N_HIJOSV,FECHA_NACM,N_EMB,SEG_SOCIAL,IDCLASADMI,EDAD_PADRE,NIV_EDUP,ULTCURPAD,PROFESION,TIPOFORMULARIO
279,44,847,3,2,NaN,1,9,9,2021,10,...,1,NaN,1,2,2.0,999,99,99,5.0,1
280,44,847,3,2,NaN,2,9,9,2021,11,...,1,NaN,1,2,2.0,999,99,99,5.0,1
281,44,847,3,2,NaN,2,9,9,2021,10,...,1,NaN,1,2,2.0,999,99,99,5.0,1
1261,44,1,1,1,NaN,1,4,4,2021,4,...,2,NaN,1,1,1.0,30,99,99,1.0,1
1317,44,430,3,2,NaN,1,9,9,2021,6,...,1,NaN,1,2,2.0,74,99,99,5.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
616810,27,787,3,2,NaN,1,9,9,2021,9,...,1,NaN,1,2,2.0,20,99,99,5.0,1
616827,27,615,1,2,NaN,2,9,9,2021,9,...,1,NaN,1,2,2.0,999,99,99,5.0,1
616851,27,615,1,2,NaN,2,9,9,2021,12,...,1,NaN,1,2,2.0,999,99,99,5.0,1
616901,27,787,3,2,NaN,1,9,9,2021,9,...,1,NaN,1,2,2.0,20,99,99,5.0,1


In [4]:
df_nac = eliminar_duplicados(df_nac) # Eliminar filas duplicadas

===== LIMPIEZA DE DUPLICADOS =====
Duplicados detectados: 698
Registros eliminados: 698
Total final: 616216


In [5]:
## Eliminación de columnas inecesarias
cols_a_borrar_nac = ['OTRO_SIT','APGAR1','APGAR2','IDPERTET','ULTCURMAD','N_HIJOSV','FECHA_NACM','N_EMB','ULTCURPAD', 'TIPOFORMULARIO', 'T_GES_AGRU_CIE']
df_nac = eliminar_columnas(df_nac, cols_a_borrar_nac)

Borradas 11 columnas de la lista especificada.


In [6]:
## Estandarización de nombres de columnas
df_nac  = estandarizar_nombres_columnas(df_nac)

In [7]:
## Manejo de valores nulos
resumen_nulos = analizar_nulos(df_nac)

===== ANÁLISIS DE NULOS =====
Total de registros: 616216
Columnas con nulos: 5



,nulos,porcentaje (%)
idclasadmi,65607,10.646754
codptore,8826,1.432290
codmunre,8826,1.432290
area_res,8823,1.431803
profesion,67,0.010873


In [8]:
df_nac[df_nac["codptore"].isnull()].shape



(8826, 28)

In [9]:
df_nac[df_nac["codmunre"].isnull()].shape


(8826, 28)

In [10]:
df_nac[df_nac["area_res"].isnull()].shape

(8823, 28)

In [11]:
df_nac[df_nac["profesion"].isnull()].shape

(67, 28)

In [12]:
# eliminar nulos geográficos
df_nac = df_nac.dropna(subset=["codptore", "codmunre", "area_res", "profesion"])

In [13]:
# 3. imputar otros nulos
estrategia = {
    "idclasadmi": {"metodo": "fill", "valor": "DESCONOCIDO"}
}

df_nac = manejar_nulos(df_nac, estrategia)

In [14]:
df_nac.isnull().sum()

cod_dpto      0
cod_munic     0
areanac       0
sit_parto     0
sexo          0
peso_nac      0
talla_nac     0
ano           0
mes           0
aten_par      0
t_ges         0
numconsul     0
tipo_parto    0
mul_parto     0
idhemoclas    0
idfactorrh    0
edad_madre    0
est_civm      0
niv_edum      0
codpres       0
codptore      0
codmunre      0
area_res      0
seg_social    0
idclasadmi    0
edad_padre    0
niv_edup      0
profesion     0
dtype: int64

In [15]:
## Ajustar tipos de datos
df_nac = ajustar_tipos_datos(df_nac)

In [16]:
## Guardar el DataFrame procesado
""" out_nac = '../data/processed/nacimientos/nacimientos_2021.parquet'
df_nac.to_parquet(out_nac, index=False)
print(f"Guardado Nacimientos 2021 en: {out_nac}") """

' out_nac = \'../data/processed/nacimientos/nacimientos_2021.parquet\'\ndf_nac.to_parquet(out_nac, index=False)\nprint(f"Guardado Nacimientos 2021 en: {out_nac}") '

In [17]:
VALID_COLUMNS_NACIMIENTOS = {
    col: str(df_nac[col].dtype) for col in VALID_COLUMNS_NACIMIENTOS
}

validar_schema(df_nac, VALID_COLUMNS_NACIMIENTOS)


=== RESULTADO VALIDACIÓN ===
✔ Columnas correctas: 28


## 2. Muertes Fetales

In [18]:
rutaDatos = '../data/raw/fetal2021.csv'
try:
    df_fet = pd.read_csv(rutaDatos, sep=',', encoding='latin-1')
    print(f"Archivo encontrado: \n")
    df_fet.info()
except FileNotFoundError:
    print(f"Archivo no encontrado: {rutaDatos}")

Archivo encontrado: 

<class 'pandas.DataFrame'>
RangeIndex: 30709 entries, 0 to 30708
Data columns (total 44 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   COD_DPTO        30709 non-null  int64  
 1   COD_MUNIC       30709 non-null  int64  
 2   A_DEFUN         30709 non-null  int64  
 3   SIT_DEFUN       30709 non-null  int64  
 4   OTRSITIODE      89 non-null     str    
 5   TIPO_DEFUN      30709 non-null  int64  
 6   ANO             30709 non-null  int64  
 7   MES             30709 non-null  int64  
 8   HORA            9982 non-null   float64
 9   MINUTOS         9982 non-null   float64
 10  SEXO            30709 non-null  int64  
 11  CODPRES         30552 non-null  float64
 12  CODPTORE        30075 non-null  float64
 13  CODMUNRE        30075 non-null  float64
 14  AREA_RES        30076 non-null  float64
 15  SEG_SOCIAL      30709 non-null  int64  
 16  IDADMISALUD     26759 non-null  float64
 17  P_PMAN_IRIS     6639

In [19]:
encontrar_duplicados(df_fet) # Encontrar filas duplicadas


===== ANÁLISIS DE DUPLICADOS =====
Total de registros: 30709
Filas duplicadas (incluyendo repetidas): 434
Duplicados únicos a eliminar: 230


,COD_DPTO,COD_MUNIC,A_DEFUN,SIT_DEFUN,OTRSITIODE,TIPO_DEFUN,ANO,MES,HORA,MINUTOS,...,C_MUERTEB,C_MUERTEC,C_MUERTED,C_MUERTEE,ASIS_MED,CAUSA_MULT,C_BAS1,CAUSA_667,IDPROFCER,CAU_HOMOL
69,11,1,1,1,NaN,1,2021,6,NaN,NaN,...,1.0,NaN,NaN,NaN,1,P018,P018,402,1,80
70,11,1,1,1,NaN,1,2021,6,NaN,NaN,...,1.0,NaN,NaN,NaN,1,P018,P018,402,1,80
76,11,1,1,1,NaN,1,2021,12,NaN,NaN,...,1.0,NaN,NaN,NaN,1,P018,P018,402,1,80
89,11,1,1,1,NaN,1,2021,12,NaN,NaN,...,1.0,NaN,NaN,NaN,1,P018,P018,402,1,80
141,54,498,1,1,NaN,1,2021,12,16.0,38.0,...,1.0,NaN,NaN,NaN,1,P018/P019/Q897,Q897,613,1,88
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30523,11,1,1,1,NaN,1,2021,4,NaN,NaN,...,1.0,NaN,NaN,NaN,1,P018,P018,402,1,80
30546,66,1,1,1,NaN,1,2021,4,NaN,NaN,...,1.0,NaN,NaN,NaN,1,P018,P018,402,1,80
30566,68,276,1,1,NaN,1,2021,5,NaN,NaN,...,1.0,NaN,NaN,NaN,1,P95,P95,406,1,86
30567,68,276,1,1,NaN,1,2021,5,NaN,NaN,...,1.0,NaN,NaN,NaN,1,P95,P95,406,1,86


In [20]:
df_fet = eliminar_duplicados(df_fet) # Eliminar filas duplicadas

===== LIMPIEZA DE DUPLICADOS =====
Duplicados detectados: 230
Registros eliminados: 230
Total final: 30479


In [21]:
cols_a_borrar_fet  = ['OTRSITIODE','HORA','MINUTOS','CODPRES','P_PMAN_IRIS','CONS_EXP','ULTCURMAD','CODOCUR','CODMUNOC','C_MUERTE','C_MUERTEB','C_MUERTEC','C_MUERTED','C_MUERTEE','CAUSA_MULT','C_BAS1','CAUSA_667','IDPROFCER', 'T_GES_AGRU_CIE']
df_fet = eliminar_columnas(df_fet, cols_a_borrar_fet)

Borradas 19 columnas de la lista especificada.


In [22]:
df_fet  = estandarizar_nombres_columnas(df_fet)

In [23]:
resumen_nulos = analizar_nulos(df_fet)

===== ANÁLISIS DE NULOS =====
Total de registros: 30479
Columnas con nulos: 4



,nulos,porcentaje (%)
idadmisalud,3924,12.874438
codmunre,627,2.057154
codptore,627,2.057154
area_res,626,2.053873


In [24]:
df_fet[df_fet["codmunre"].isnull()].shape

(627, 25)

In [25]:
df_fet[df_fet["area_res"].isnull()].shape

(626, 25)

In [26]:
df_fet[df_fet["codptore"].isnull()].shape

(627, 25)

In [27]:
df_fet = df_fet.dropna(subset=["codptore", "codmunre", "area_res"])

In [28]:
estrategia = {
    "idadmisalud": {"metodo": "fill", "valor": "DESCONOCIDO"}
}

df_fet = manejar_nulos(df_fet, estrategia)

In [29]:
df_fet.isnull().sum()

cod_dpto       0
cod_munic      0
a_defun        0
sit_defun      0
tipo_defun     0
ano            0
mes            0
sexo           0
codptore       0
codmunre       0
area_res       0
seg_social     0
idadmisalud    0
mu_parto       0
t_parto        0
tipo_emb       0
t_ges          0
peso_nac       0
edad_madre     0
n_hijosv       0
n_hijosm       0
est_civm       0
niv_edum       0
asis_med       0
cau_homol      0
dtype: int64

In [30]:
df_fet = ajustar_tipos_datos(df_fet)

In [31]:
## Guardar el DataFrame procesado
""" out_fet = '../data/processed/fetales/fetales2021.parquet'
df_fet.to_parquet(out_fet, index=False)
print(f"Guardado Fetales 2021 en: {out_fet}") """

' out_fet = \'../data/processed/fetales/fetales2021.parquet\'\ndf_fet.to_parquet(out_fet, index=False)\nprint(f"Guardado Fetales 2021 en: {out_fet}") '

In [32]:
VALID_COLUMNS_FETALES = {
    col: str(df_fet[col].dtype) for col in VALID_COLUMNS_FETALES
}

validar_schema(df_fet, VALID_COLUMNS_FETALES)


=== RESULTADO VALIDACIÓN ===
✔ Columnas correctas: 23

➕ Columnas extra (2): ['seg_social', 'idadmisalud']


In [33]:
# Podemos observar que tenemso 'seg_social', 'idadmisalud' extras en df_fet, las vamos a borrar.
# Debido a que en el 2024 estos datos no se encuentran disponibles, no tiene sentido mantenerlos en el esquema. Además, esto nos permitirá tener un esquema consistente a lo largo de los años, facilitando el análisis comparativo.

cols_a_borrar_fet_extra = ['seg_social', 'idadmisalud']
df_fet = eliminar_columnas(df_fet, cols_a_borrar_fet_extra)

Borradas 2 columnas de la lista especificada.


In [34]:
validar_schema(df_fet, VALID_COLUMNS_FETALES)


=== RESULTADO VALIDACIÓN ===
✔ Columnas correctas: 23


## 3. No Fetales 

In [35]:
rutaDatos = '../data/raw/nofetal2021.csv'
try:
    df_nofet = pd.read_csv(rutaDatos, sep=',', encoding='latin-1')
    print(f"Archivo encontrado: \n")
    df_nofet.info()
except FileNotFoundError:
    print(f"Archivo no encontrado: {rutaDatos}")

Archivo encontrado: 

<class 'pandas.DataFrame'>
RangeIndex: 363089 entries, 0 to 363088
Data columns (total 56 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   COD_DPTO        363089 non-null  int64  
 1   COD_MUNIC       363089 non-null  int64  
 2   A_DEFUN         363089 non-null  int64  
 3   SIT_DEFUN       363089 non-null  int64  
 4   OTRSITIODE      7399 non-null    str    
 5   TIPO_DEFUN      363089 non-null  int64  
 6   ANO             363089 non-null  int64  
 7   MES             363089 non-null  int64  
 8   HORA            359099 non-null  float64
 9   MINUTOS         359099 non-null  float64
 10  SEXO            363089 non-null  int64  
 11  EST_CIVIL       363089 non-null  int64  
 12  GRU_ED1         363089 non-null  int64  
 13  GRU_ED2         363089 non-null  int64  
 14  NIVEL_EDU       363089 non-null  int64  
 15  ULTCURFAL       363089 non-null  int64  
 16  MUERTEPORO      355106 non-null  float64
 17 

In [36]:
encontrar_duplicados(df_nofet)

===== ANÁLISIS DE DUPLICADOS =====
Total de registros: 363089
Filas duplicadas (incluyendo repetidas): 89
Duplicados únicos a eliminar: 47


,COD_DPTO,COD_MUNIC,A_DEFUN,SIT_DEFUN,OTRSITIODE,TIPO_DEFUN,ANO,MES,HORA,MINUTOS,...,C_MUERTEB,C_MUERTEC,C_MUERTED,C_MUERTEE,ASIS_MED,CAUSA_MULT,C_BAS1,CAUSA_667,IDPROFCER,CAU_HOMOL
264,47,720,3,3,NaN,2,2021,8,NaN,NaN,...,NaN,NaN,1.0,NaN,2,R98,R98,0,5,89
2900,5,360,1,3,NaN,2,2021,6,3.0,0.0,...,NaN,NaN,1.0,NaN,2,C799/C61,C61,210,1,28
3660,47,720,3,3,NaN,2,2021,8,NaN,NaN,...,NaN,NaN,1.0,NaN,2,R98,R98,0,5,89
6082,11,1,1,3,NaN,2,2021,4,0.0,0.0,...,NaN,NaN,NaN,NaN,2,G931/T71/T71 X70,X700,511,1,100
9643,18,410,2,9,NaN,2,2021,10,0.0,0.0,...,NaN,NaN,NaN,NaN,2,S062/S018/T141 X95,X959,512,1,101
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
336701,76,1,1,1,NaN,2,2021,9,6.0,0.0,...,1.0,NaN,NaN,NaN,1,I219/I509/I10,I219,303,1,51
352206,52,79,3,9,NaN,2,2021,1,NaN,NaN,...,NaN,NaN,1.0,NaN,2,R98,R98,0,1,89
354609,19,397,3,3,NaN,2,2021,8,6.0,0.0,...,NaN,NaN,1.0,NaN,2,I219,I219,303,4,51
355589,70,708,1,3,NaN,2,2021,2,0.0,0.0,...,NaN,NaN,1.0,NaN,2,R98,R98,0,5,89


In [37]:
df_nofet = eliminar_duplicados(df_nofet)

===== LIMPIEZA DE DUPLICADOS =====
Duplicados detectados: 47
Registros eliminados: 47
Total final: 363042


In [38]:
cols_a_borrar_nofet = ['OTRSITIODE','HORA','MINUTOS','CODPRES','P_PMAN_IRIS','CONS_EXP','ULTCURMAD','CODOCUR','CODMUNOC','C_MUERTE','C_MUERTEB','C_MUERTEC','C_MUERTED','C_MUERTEE', 'CAUSA_MULT','C_BAS1','CAUSA_667','IDPROFCER','MU_PARTO','CAU_HOMOL', 'T_GES_AGRU_CIE']
df_nofet = eliminar_columnas(df_nofet, cols_a_borrar_nofet)

Borradas 21 columnas de la lista especificada.


In [39]:
df_nofet = estandarizar_nombres_columnas(df_nofet)

In [40]:
resumen_nulos = analizar_nulos(df_nofet)

===== ANÁLISIS DE NULOS =====
Total de registros: 363042
Columnas con nulos: 19



,nulos,porcentaje (%)
simuertepo,362567,99.869161
n_hijosm,356292,98.140711
est_civm,356292,98.140711
n_hijosv,356292,98.140711
t_parto,356292,98.140711
peso_nac,356292,98.140711
edad_madre,356292,98.140711
tipo_emb,356292,98.140711
t_ges,356292,98.140711
niv_edum,356292,98.140711


In [41]:
df_nofet = df_nofet.dropna(subset=["codptore", "codmunre", "area_res"])

In [42]:
estrategia = {
    "ocupacion": {"metodo": "fill", "valor": "DESCONOCIDO"},
    "muerteporo": {"metodo": "fill_mode"},
    "idadmisalud": {"metodo": "fill", "valor": "DESCONOCIDO"}
}

df_nofet = manejar_nulos(df_nofet, estrategia)

In [43]:
columnas_eliminar = [
    "simuertepo",
    "t_ges",
    "niv_edum",
    "n_hijosm",
    "tipo_emb",
    "t_parto",
    "est_civm",
    "peso_nac",
    "edad_madre",
    "n_hijosv",
    "emb_mes",
    "emb_sem",
    "emb_fal"
]

df_nofet = eliminar_columnas(df_nofet, columnas_eliminar)

Borradas 13 columnas de la lista especificada.


In [44]:
df_nofet = ajustar_tipos_datos(df_nofet)

In [45]:
## Guardar el DataFrame procesado
""" out_fet = '../data/processed/fetales/fetales2021.parquet'
df_fet.to_parquet(out_fet, index=False)
print(f"Guardado Fetales 2021 en: {out_fet}") """

' out_fet = \'../data/processed/fetales/fetales2021.parquet\'\ndf_fet.to_parquet(out_fet, index=False)\nprint(f"Guardado Fetales 2021 en: {out_fet}") '

In [46]:
VALID_COLUMNS_NO_FETALES = {
    col: str(df_nofet[col].dtype) for col in VALID_COLUMNS_NO_FETALES
}
validar_schema(df_nofet, VALID_COLUMNS_NO_FETALES)


=== RESULTADO VALIDACIÓN ===
✔ Columnas correctas: 22


In [47]:
VALID_COLUMNS_NO_FETALES


{'a_defun': 'int8',
 'ano': 'int16',
 'area_res': 'float32',
 'asis_med': 'int8',
 'cod_dpto': 'int8',
 'cod_munic': 'int16',
 'codmunre': 'float32',
 'codptore': 'float32',
 'est_civil': 'int8',
 'gru_ed1': 'int8',
 'gru_ed2': 'int8',
 'idadmisalud': 'object',
 'idpertet': 'int8',
 'mes': 'int8',
 'muerteporo': 'float32',
 'nivel_edu': 'int8',
 'ocupacion': 'str',
 'seg_social': 'int8',
 'sexo': 'int8',
 'sit_defun': 'int8',
 'tipo_defun': 'int8',
 'ultcurfal': 'int8'}

In [48]:
set(df_nofet.columns)

{'a_defun',
 'ano',
 'area_res',
 'asis_med',
 'cod_dpto',
 'cod_munic',
 'codmunre',
 'codptore',
 'est_civil',
 'gru_ed1',
 'gru_ed2',
 'idadmisalud',
 'idpertet',
 'mes',
 'muerteporo',
 'nivel_edu',
 'ocupacion',
 'seg_social',
 'sexo',
 'sit_defun',
 'tipo_defun',
 'ultcurfal'}